In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import imblearn
import warnings

from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('Autism prediction dataset test.csv') # Adjusted filename for local use
df.head()

# EDA
df.shape
df.duplicated().sum()
df.isnull().sum()
df.info()

# Preprocessing
df.columns
df['Class/ASD'].value_counts()

# The data in target column is imbalanced. So we will now oversample it
from sklearn.utils import resample
df_majority = df[(df['Class/ASD']==0)]
df_minority = df[(df['Class/ASD']==1)]

# upsample minority class
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=639,    # to match majority class
                                 random_state=42) # reproducible results

# Combine majority class with upsampled minority class
df = pd.concat([df_minority_upsampled, df_majority])

df['Class/ASD'].value_counts()
df.shape

# Cleaning
df = df[df['result'] > -5]
df.shape

# Feature Engineering
def convertAge(age):
    if age < 4:
        return 'Baby'
    elif age < 12:
        return 'Kid'
    elif age < 18:
        return 'Teenager'
    elif age < 40:
        return 'Young'
    else:
        return 'Senior/OLD'

df['ageGroup'] = df['age'].apply(convertAge)

def add_features(data):
    #creating a column with all values zero
    df['sum_score'] = 0
    for col in df.loc[:, 'A1_Score':'A10_Score'].columns:
        df['sum_score'] += df[col]
        
    #Now creating a random data using the below columns
    df['Pak'] = df['austim'] + df['used_app_before'] + df['jundice']
    
    return data

df = add_features(df)

# Encoding
from sklearn.preprocessing import LabelEncoder

def encode_labels(data):
    for col in data.columns:
        if df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
    return data

df = encode_labels(df)

# Correlation
corr = df.corr()
plt.figure(figsize=(25,9))
sns.heatmap(corr, annot=True, cbar=True, cmap='plasma')
plt.show()

# Split and Scale
x = df.drop(['ID', 'used_app_before', 'Class/ASD'], axis=1)
y = df['Class/ASD']

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

lr = LogisticRegression()
rf = RandomForestClassifier()
gb = GradientBoostingClassifier()
xgb = XGBClassifier()
svc = SVC()
knn = KNeighborsClassifier()
nb = GaussianNB()
lgb = LGBMClassifier()
cat = CatBoostClassifier()

# Fitting
lr.fit(x_train_scaled, y_train)
rf.fit(x_train_scaled, y_train)
gb.fit(x_train_scaled, y_train)
xgb.fit(x_train_scaled, y_train)
svc.fit(x_train_scaled, y_train)
knn.fit(x_train_scaled, y_train)
nb.fit(x_train_scaled, y_train)

lgb.set_params(verbosity=-1)
lgb.fit(x_train_scaled, y_train)
cat.fit(x_train_scaled, y_train, verbose=False)

# Predictions
lrpred = lr.predict(x_test_scaled)
rfpred = rf.predict(x_test_scaled)
gbpred = gb.predict(x_test_scaled)
xgbpred = xgb.predict(x_test_scaled)
svcpred = svc.predict(x_test_scaled)
knnpred = knn.predict(x_test_scaled)
nbpred = nb.predict(x_test_scaled)
lgbpred = lgb.predict(x_test_scaled)
catpred = cat.predict(x_test_scaled)

# Evaluation
from sklearn.metrics import accuracy_score
print('LOGISTIC REG', accuracy_score(y_test, lrpred))
print('RANDOM FOREST', accuracy_score(y_test, rfpred))
print('GB', accuracy_score(y_test, gbpred))
print('XGB', accuracy_score(y_test, xgbpred))
print('SVC', accuracy_score(y_test, svcpred))
print('KNN', accuracy_score(y_test, knnpred))
print('NB', accuracy_score(y_test, nbpred))
print('LIGHT GBM', accuracy_score(y_test, lgbpred))
print('CATO', accuracy_score(y_test, catpred))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, rfpred)
plt.title('Heatmap of Confusion matrix', fontsize=15)
sns.heatmap(cm, annot=True)
plt.show()

# Cross Validation
from sklearn.model_selection import cross_val_score
cross_val = cross_val_score(estimator=svc, X=x_train_scaled, y=y_train)
print('Cross Val Acc Score of SVC model is ---> ', cross_val)
print('\n Cross Val Mean Acc Score of SVC model is ---> ', cross_val.mean())
